In [ ]:
import pandas as pd
import plotly.express as px

### 데이터 확인

확인내용
- 앞에서 30까지의 변수는 결함 외 변수
- 31개부터는 결함 변수

조치내용
- 실제 컬럼명을 포함하는 2번째 행부터 데이터 읽기
- **문제를 단순화하기 위해 결함 유형을 한 개로 축소**

In [ ]:
df = pd.read_csv("../data/DieCasting_Quality_Raw_Data.csv")
display(df.head())
print(df.columns)
print(df.columns[:31])
print(df.columns[31:])

In [ ]:
df = pd.read_csv("../data/DieCasting_Quality_Raw_Data.csv", skiprows=1)
df["defect"] = df[df.columns[31:]].max(axis=1).astype(bool).astype(int)
df = df[df.columns[:31].tolist() + ["defect"]]

In [ ]:
# Shape and data types
display(df.shape)

### ID 컬럼 확인
확인내용
- 행의 개수와 ID컬럼 unique한 수가 같음(즉, 각 행별로 ID컬럼이 다른 값을 가짐)
- 기존 문서에는 설비ID라고 명시되어 있으나, 잘못된 내용이거나 흔히 생각하는 설비ID와는 다를 것으로 보임
  (일반적으로 설비ID라고 하면, 공장 내에 존재하는 설비의 식별자를 의미함)
- 데이터 순서별로 ID를 그리면 점차 증가하는 경향을 나타냄. 설비ID보다는 새로 생산되면서 하나씩 늘어나는 제품ID에 가까운 값일 가능성이 높음

조치내용
- ID컬럼을 Feature로 활용하지 않음

In [ ]:
print(df.shape)
print(df["id"].nunique())
px.line(x=df.index, y=df["id"], width=500, height=300)

### 결측값 확인

확인내용
- Factory_Temp 및 Factory_Humidity 관련 변수에서 결측값 발생
- 각 변수들이 동시에 90개 행에 대해 결측값이 발생함

조치내용
- 결측값이 포함된 행 제외

In [ ]:
# 행 결측값 확인
missing_rows = df.isnull().sum(axis=1)
display(missing_rows[missing_rows > 0])
# 열 결측값 확인
missing_cols = df.isnull().sum()
display(missing_cols[missing_cols > 0])
display(f"결측값 제거 전:{df.shape}")
df = df[missing_rows == 0]
display(f"결측값 제거 후:{df.shape}")

### 기본 통계확인

확인내용
- 일부 컬럼이 항상 같은 값을 가짐(std=0)
   - ['Air_Pressure_Min', 'Air_Pressure_Max', 'Coolant_Temp_Min', 'Coolant_Temp_Max', 'Factory_Temp_Min', 'Factory_Temp_Max', 'Factory_Humidity_Min', 'Factory_Humidity_Max']

조치내용
- 해당 컬럼 제외

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
display(df.describe())
# Find columns with zero standard deviation
zero_std_cols = df.describe().loc["std"][df.describe().loc["std"] == 0].index.tolist()
print("Columns with zero standard deviation:")
print(zero_std_cols)

df = df.drop(columns=zero_std_cols)


### 결함 컬럼 분포 확인
확인내용
- 결함이 1689행, 양품이 5846 행이며, 이는 라벨링을 위해 샘플링 됐을 가능성이 높음
- 실제 분포는 이와는 다를 가능성이 높으며(불량률 22%는 상식적으로 너무 큼), 현재의 데이터만으로는 실제 모델 배포시 과검율을 제대로 확인할 수 없음
- (샘플링 했음에도 불구하고) 양품이 결함에 비해 높음. 모델링 시, 라벨 불균형에 대한 조치가 필요할 수 있음

조치내용
- 특정 기간에 대한 전체 생산 데이터 요청

In [ ]:
fig_target = px.histogram(
    df, x="defect", title="Target Distribution: defect", width=400, height=400
)
fig_target.update_layout(bargap=0.2)  # Adjust gap between bars if needed
fig_target.show()

### 결함 양/불 별 변수 분포 확인
확인내용
- 각 변수들이 양/불 별로 약간의 분포 차이를 보임
- 양/불을 결정하는 하나의 매우 영향력이 큰 변수가 있어보이지는 않음

In [ ]:
# 수치형 컬럼 선택 (id, Shot 제외)
numeric_cols = (
    df.select_dtypes(include=["int64", "float64"])
    .drop(columns=["id", "Shot"], errors="ignore")
    .columns.tolist()
)
df_melted = df.melt(
    id_vars="defect", value_vars=numeric_cols, var_name="variable", value_name="value"
)

# plotly로 boxplot 그리기
fig = px.box(
    df_melted,
    x="defect",
    y="value",
    facet_col="variable",
    facet_col_wrap=len(numeric_cols) // 5 + 1,
    color="defect",
    boxmode="group",
    height=1200,
    width=1600,
    title="Boxplots of Variables by Defect",
)

fig.update_layout(showlegend=True)
fig.update_yaxes(matches=None, showticklabels=True)
fig.show()

In [ ]:
# 수치형 컬럼 선택 (id, Shot 제외)
numeric_cols = (
    df.select_dtypes(include=["int64", "float64"])
    .drop(columns=["id", "Shot"], errors="ignore")
    .columns.tolist()
)
df_melted = df.melt(
    id_vars="defect", value_vars=numeric_cols, var_name="variable", value_name="value"
)


# Function to remove outliers based on IQR
def remove_outliers_iqr(df, col="value"):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    return df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]


# Apply filtering per variable
filtered_df = pd.concat(
    [remove_outliers_iqr(group) for _, group in df_melted.groupby("variable")]
)

# plotly로 boxplot 그리기
fig = px.box(
    filtered_df,
    x="defect",
    y="value",
    facet_col="variable",
    facet_col_wrap=len(numeric_cols) // 5 + 1,
    color="defect",
    boxmode="group",
    height=1200,
    width=1600,
    title="Boxplots of Variables by Defect",
)

fig.update_layout(showlegend=True)
fig.update_yaxes(matches=None, showticklabels=True)
fig.show()